# Lab 2E: Build a RAG assistant on company documents

**Presenter: Nagarjuna Naidu**  |  Day 2, AI Solution Architect program

**Learning outcomes**
- Ingest, chunk and index a document set with metadata (embeddings + Chroma)
- Retrieve safely: role-based security trimming, freshness rules, "no confident answer" threshold
- Generate cited answers, and measure retrieval with a mini evaluation

**Time budget:** ~35 min core (sections 1-7). Section 8 is OPTIONAL for spare time.

## 1. Setup

In [ ]:
import sys, os, re, json, glob, shutil, random, warnings, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
random.seed(7); np.random.seed(7)

INK, TEAL, LIME, CORAL, GOLD, PURPLE = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134", "#6C4AB6"
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=[TEAL, CORAL, GOLD, PURPLE, LIME, INK]),
                     "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
pathlib.Path("artifacts").mkdir(exist_ok=True)

sys.path.insert(0, ".")
from llm_client import call_llm, llm_available, llm_mode
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)
print("LLM mode:", llm_mode())

**Connect a free LLM (optional)** - set these env vars *before* starting Jupyter:

```
LLM_API_KEY=<your-key>            LLM_MODEL=<model name>
LLM_BASE_URL=<OpenAI-compatible URL>
  Groq:               https://api.groq.com/openai/v1
  Google AI Studio:   https://generativelanguage.googleapis.com/v1beta/openai/
  Local Ollama:       http://localhost:11434/v1   (no key needed)
Azure OpenAI: AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, AZURE_OPENAI_API_VERSION, LLM_MODEL=<deployment>
```
- **No key? No problem.** The lab runs in *offline demo mode*: the "LLM" quotes the best-matching sentences with citations.
  Retrieval, filtering and evaluation are identical either way; only the wording of the answer differs.

## 2. Ingest: parse, chunk, embed, store

In [ ]:
def parse_doc(path):
    """Header lines (key: value) up to the first blank line, then the body."""
    head, body = pathlib.Path(path).read_text().split("\n\n", 1)
    meta = {k.strip(): v.strip() for k, v in (l.split(":", 1) for l in head.splitlines() if ":" in l)}
    meta["doc_id"] = re.match(r"(D\d+)", pathlib.Path(path).name).group(1)
    meta.setdefault("policy_id", "")
    return meta, body.strip()

DOCS = [parse_doc(p) for p in sorted(glob.glob("data/corpus/*.txt"))]
print(len(DOCS), "documents;", "sensitivity:", pd.Series([m["sensitivity"] for m, _ in DOCS]).value_counts().to_dict())

# How are versions marked? Same policy_id, different date (no explicit "superseded" field).
pol = pd.DataFrame([m for m, _ in DOCS]); pol = pol[pol.policy_id != ""]
dup = pol[pol.duplicated("policy_id", keep=False)].sort_values(["policy_id", "date"])
print("\nPolicy IDs that appear in more than one document:")
display(dup[["doc_id", "policy_id", "date", "title"]])

In [ ]:
MAX_WORDS = 170

def sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()]

def chunk_body(body, max_words=MAX_WORDS):
    """Paragraph/heading-aware chunks (<= max_words) with a 1-sentence overlap between neighbours."""
    paras = [p.strip() for p in re.split(r"\n\s*\n", body) if p.strip()]
    units, i = [], 0
    while i < len(paras):                      # a short heading line sticks to the paragraph after it
        p = paras[i]
        if len(p.split()) <= 6 and "\n" not in p and i + 1 < len(paras):
            p = p + "\n" + paras[i + 1]; i += 1
        units.append(p); i += 1
    chunks, cur = [], ""
    for u in units:
        if cur and len((cur + " " + u).split()) > max_words:
            chunks.append(cur)
            tail = sentences(cur)[-1]                       # overlap = last sentence of previous chunk
            cur = tail + "\n" + u
        else:
            cur = (cur + "\n\n" + u) if cur else u
    if cur: chunks.append(cur)
    out = []                                                # split any single over-long unit by sentences
    for c in chunks:
        if len(c.split()) <= max_words * 1.5: out.append(c); continue
        buf = ""
        for s in sentences(c):
            if buf and len((buf + " " + s).split()) > max_words:
                out.append(buf); buf = s
            else: buf = (buf + " " + s).strip()
        if buf: out.append(buf)
    return out

RECORDS = []
for meta, body in DOCS:
    for j, text in enumerate(chunk_body(body)):
        RECORDS.append(dict(id=f"{meta['doc_id']}-c{j}", text=text,
                            meta=dict(doc_id=meta["doc_id"], title=meta["title"], department=meta["department"],
                                      date=meta["date"], sensitivity=meta["sensitivity"],
                                      policy_id=meta["policy_id"], chunk_id=f"{meta['doc_id']}-c{j}")))
wc = pd.Series([len(r["text"].split()) for r in RECORDS])
print(f"{len(RECORDS)} chunks from {len(DOCS)} docs | words/chunk: min {wc.min()}, median {int(wc.median())}, max {wc.max()}")
print("\nExample chunk:", RECORDS[0]["meta"]["chunk_id"], "\n", RECORDS[0]["text"][:350], "...")

In [ ]:
from fastembed import TextEmbedding
import chromadb

embedder = TextEmbedding("BAAI/bge-small-en-v1.5")
def embed(texts):
    return [v.tolist() for v in embedder.embed(list(texts))]

CHROMA_DIR = "artifacts/rag_chroma"
shutil.rmtree(CHROMA_DIR, ignore_errors=True)                       # fresh index every run
client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.create_collection("northwind_docs", metadata={"hnsw:space": "cosine"})
# embed "title + text" so a chunk knows which document it came from; store the raw text for display
vecs = embed(f"{r['meta']['title']}. {r['text']}" for r in RECORDS)
col.add(ids=[r["id"] for r in RECORDS], documents=[r["text"] for r in RECORDS],
        embeddings=vecs, metadatas=[r["meta"] for r in RECORDS])
print("indexed", col.count(), "chunks | dim", len(vecs[0]), "| stored in", CHROMA_DIR)
print(pd.DataFrame([r["meta"] for r in RECORDS]).groupby("sensitivity").chunk_id.count().to_dict())

## 3. Retrieve: vector search + security trimming + freshness + "no answer" threshold
- **Security trimming** is applied *inside* the vector query (`where=`), so forbidden chunks are never even candidates.
- **Freshness**: if two documents share a `policy_id`, only the newest is searchable by default.
- **Threshold**: if even the best chunk is not similar enough, say so instead of answering.

In [ ]:
# Role -> Chroma `where` fragment (None = no restriction). Unknown roles are DENIED (fail closed).
ROLE_RULES = {
    "intern":     {"sensitivity": {"$in": ["public"]}},
    "employee":   {"sensitivity": {"$in": ["public", "internal"]}},
    "manager":    None,
    "hr_partner": None,
}

# Freshness: for each policy_id keep only the document with the latest date.
_latest = pol.groupby("policy_id").date.max().to_dict()
SUPERSEDED = sorted(pol[pol.date < pol.policy_id.map(_latest)].doc_id)
print("Superseded documents (hidden by default):", SUPERSEDED)

MIN_SIM = 0.0   # no-answer threshold on top-1 cosine similarity; calibrated below

def retrieve(question, k=5, role="employee", fresh=True, min_sim=None):
    if role not in ROLE_RULES:
        raise ValueError(f"unknown role '{role}': access denied")
    clauses = [c for c in (ROLE_RULES[role],) if c]
    if fresh and SUPERSEDED:
        clauses.append({"doc_id": {"$nin": SUPERSEDED}})
    where = None if not clauses else clauses[0] if len(clauses) == 1 else {"$and": clauses}
    res = col.query(query_embeddings=embed([question]), n_results=k, where=where)
    chunks = [dict(text=t, sim=round(1 - d, 3), **m)
              for t, m, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]
    thr = MIN_SIM if min_sim is None else min_sim
    top = chunks[0]["sim"] if chunks else 0.0
    return dict(chunks=chunks, top_sim=top, confident=bool(chunks) and top >= thr)

r = retrieve("how much time off do I get when my baby is born", k=3, role="employee")
for c in r["chunks"]: print(c["sim"], c["doc_id"], c["date"], c["title"])

In [ ]:
# Calibrate the threshold on the gold queries (top-1 similarity: answerable vs no_answer)
EVAL = json.load(open("data/eval_queries.json"))
cal = pd.DataFrame([dict(query=e["query"], type=e["type"], top_sim=retrieve(e["query"], 5, "hr_partner", min_sim=0)["top_sim"])
                    for e in EVAL])
cal["answerable"] = cal.type != "no_answer"
print(cal.groupby("type").top_sim.agg(["count", "min", "median", "max"]).round(3))

grid = np.arange(0.40, 0.80, 0.02)
def outcome(t):
    return dict(threshold=round(float(t), 2),
                answerable_refused=int(((cal.top_sim < t) & cal.answerable).sum()),
                no_answer_refused=int(((cal.top_sim < t) & ~cal.answerable).sum()))
display(pd.DataFrame([outcome(t) for t in grid if round(t, 2) in (0.50, 0.54, 0.56, 0.58, 0.60, 0.64, 0.68, 0.70)]).set_index("threshold").T)

# Operating point: never refuse an answerable gold query (wrong refusals annoy users), with a small safety buffer.
MIN_SIM = round(float(cal[cal.answerable].top_sim.min()) - 0.02, 2)
print(f"\nChosen MIN_SIM = {MIN_SIM}  (lowest answerable top-1 similarity minus 0.02)")
print("Honest reading: the 4 no_answer queries are NEAR-MISSES (on-topic words, no answer) and score as high as real questions,")
print("so one similarity threshold cannot catch them without refusing many valid questions. It catches clearly off-topic queries;")
print("near-misses are the job of the prompt rule + groundedness checks (Day 3). Also: tuned and scored on the SAME queries = optimistic.")

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.scatter(cal[cal.answerable].top_sim, np.random.rand((cal.answerable).sum()), c=TEAL, label="answerable")
ax.scatter(cal[~cal.answerable].top_sim, np.random.rand((~cal.answerable).sum()), c=CORAL, label="no_answer")
ax.axvline(MIN_SIM, c=INK, ls="--"); ax.set_yticks([]); ax.set_xlabel("top-1 cosine similarity"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 4. Generate: numbered context + citations

In [ ]:
SYSTEM = ("You are a careful assistant for company staff. Answer ONLY from the numbered context. "
          "Cite the sources you used like [S1] or [S2]. "
          "If the context does not contain the answer, reply exactly: I could not find this in the provided documents.")
REFUSAL = "I could not find this in the provided documents."

def build_prompt(question, chunks):
    ctx = "\n\n".join(f"[S{i}] {c['text']}" for i, c in enumerate(chunks, 1))
    return f"{ctx}\n\nQUESTION: {question}"

def answer(question, role="employee", k=4, system=SYSTEM):
    r = retrieve(question, k=k, role=role)
    chunks = r["chunks"]
    if not r["confident"]:                              # below threshold: do not even call the LLM
        return dict(answer=REFUSAL, citations=[], chunks=chunks, confident=False, top_sim=r["top_sim"])
    prompt = build_prompt(question, chunks)
    if not llm_available():   # offline stand-in finds the FIRST word "question" in the prompt; keep chunk text from tripping it
        prompt = re.sub(r"(?i)question(?=s?\b)", "query", prompt.rsplit("\n\nQUESTION:", 1)[0]) + "\n\nQUESTION:" + prompt.rsplit("\n\nQUESTION:", 1)[1]
    try:
        text = call_llm(prompt, system=system, temperature=0.0, max_tokens=400)
    except Exception as e:                              # defensive: never crash a live demo
        text = f"[LLM error: {type(e).__name__}] " + REFUSAL
    cited = sorted({int(n) for n in re.findall(r"\[S(\d+)\]", text) if 1 <= int(n) <= len(chunks)})
    cites = [dict(tag=f"S{n}", title=chunks[n-1]["title"], doc_id=chunks[n-1]["doc_id"]) for n in cited]
    return dict(answer=text, citations=cites, chunks=chunks, confident=True, top_sim=r["top_sim"])

import textwrap
def show(question, role="employee", **kw):
    out = answer(question, role, **kw)
    print(f"Q ({role}): {question}")
    print(textwrap.fill("A: " + out["answer"], 110, subsequent_indent="   "))
    print("   citations:", [f"{c['tag']}={c['doc_id']} {c['title']}" for c in out["citations"]] or "none")
    print("   retrieved:", [(c["doc_id"], c["sim"]) for c in out["chunks"]], "| top_sim", out["top_sim"])
    print()
    return out

print(build_prompt("DEMO QUESTION", [dict(text="First chunk text."), dict(text="Second chunk text.")]))

## 5. Demo run: five showcase questions

In [ ]:
print("1) Semantic question - the doc never says 'Spain'")
_ = show("Can I work from Spain for a month?", "employee")
print("2) Versioned policy - must use the NEWER (2024) policy")
_ = show("How many weeks of parental leave does the primary carer get?", "employee")
r_old = retrieve("How many weeks of parental leave does the primary carer get?", 3, "employee", fresh=False)
print("   (without the freshness rule the top hits would be:", [c["doc_id"] for c in r_old["chunks"]], ")\n")

In [ ]:
print("3) Access control - same question, two roles (the source is a CONFIDENTIAL lessons-learned document)")
q = "What went wrong in the Falcon go-live cutover?"
_ = show(q, "intern")
_ = show(q, "hr_partner")

In [ ]:
print("4) Exact-ID question")
_ = show("INC-22007", "employee")
print("5) Out of scope")
_ = show("What is the best recipe for chocolate cake?", "employee")
print("5b) Near-miss (on topic, but no document answers it) - the similarity threshold does NOT catch this one:")
_ = show("What is the dress code for client meetings?", "employee")

## 6. Mini evaluation (retrieval only; RAG evaluation goes much deeper on Day 3)
- Recall@k at **document level**: fraction of the relevant docs found in the top-k chunks.
- Superseded docs are removed from the gold list (we hide them on purpose).
- Run as `hr_partner` so access rules do not hide gold documents.

In [ ]:
rows, refusals = [], []
for e in EVAL:
    r = retrieve(e["query"], k=5, role="hr_partner")
    if e["type"] == "no_answer":
        refusals.append(not r["confident"]); continue
    gold = set(e["relevant_doc_ids"]) - set(SUPERSEDED)
    ranked = []
    for c in r["chunks"]:
        if c["doc_id"] not in ranked: ranked.append(c["doc_id"])
    rows.append(dict(type=e["type"], r3=len(gold & set(ranked[:3])) / len(gold), r5=len(gold & set(ranked)) / len(gold)))
ev = pd.DataFrame(rows)
tab = ev.groupby("type")[["r3", "r5"]].mean().round(2)
tab.loc["ALL answerable"] = ev[["r3", "r5"]].mean().round(2)
tab.columns = ["Recall@3", "Recall@5"]
display(tab)
print(f"No-answer correct refusals: {sum(refusals)}/{len(refusals)} (threshold tuned on these same queries - optimistic)")
print("Answerable queries wrongly refused:",
      sum(1 for e in EVAL if e["type"] != "no_answer" and not retrieve(e["query"], 5, "hr_partner")["confident"]), "of", len(rows))
print("\nNote: k=5 chunks can come from fewer than 5 documents, so doc-level Recall@5 is a strict measure here.")

## 7. Your turn
**Turn 1.** Change the system prompt so *every claim is followed by a short quote* from the source.

In [ ]:
SYSTEM_QUOTES = SYSTEM   # TODO: extend this prompt (hint: "After each claim add a short verbatim quote in quotation marks, then the [S#] tag")
out = answer("How many days per week do I have to be in the office?", "employee", system=SYSTEM_QUOTES)
print(out["answer"])

**Turn 2.** Add a role `contractor`: public documents, plus *internal* documents from the **IT** department only.
Unknown roles are denied by default (fail closed) - see what happens first.

In [ ]:
try:
    retrieve("How do I set up the VPN?", 3, "contractor")
except ValueError as err:
    print("Denied as expected:", err)
# TODO: add ROLE_RULES["contractor"] = ...   (hint: {"$or": [ {...public...}, {"$and": [ {...internal...}, {...department IT...} ]} ]})

**Turn 3 (OPTIONAL).** Ask your own question, as any role.

In [ ]:
my_question = "Who approves a software purchase for my team?"
my_role = "employee"
_ = show(my_question, my_role)

## 8. OPTIONAL stretch (spare time / homework)
### 8a. Re-rank by lexical overlap
Fetch more candidates (k=10), then blend vector similarity with word overlap. Helps IDs and rare terms.

In [ ]:
def toks(s): return set(re.findall(r"[a-z0-9\-]+", s.lower()))
def retrieve_reranked(question, k=5, role="hr_partner", pool=10, w=0.3):
    r = retrieve(question, pool, role, min_sim=0)
    q = toks(question)
    for c in r["chunks"]:
        c["score"] = c["sim"] + w * len(q & toks(c["text"])) / max(1, len(q))
    r["chunks"] = sorted(r["chunks"], key=lambda c: -c["score"])[:k]
    return r

def doc_recall(fn, types=("keyword", "hybrid"), k=3):
    vals = []
    for e in EVAL:
        if e["type"] in types:
            gold = set(e["relevant_doc_ids"]) - set(SUPERSEDED)
            got = {c["doc_id"] for c in fn(e["query"])["chunks"][:k]}
            vals.append(len(gold & got) / len(gold))
    return round(float(np.mean(vals)), 2)
print("keyword+hybrid Recall@3 - plain vector:", doc_recall(lambda q: retrieve(q, 5, "hr_partner", min_sim=0)),
      "| reranked:", doc_recall(retrieve_reranked))
print("(Only 10 queries: treat the difference as a hint, not a result. Day 3 covers proper hybrid search and rerankers.)")

### 8b. What would you monitor in production?
- **Retrieval quality**: Recall@k on a refreshed gold set; share of queries answered vs refused
- **Groundedness**: sampled answers checked for claims not backed by the cited chunk; citation validity
- **Security**: access-denied counts, audits that no restricted chunk reached a lower-privileged user
- **Freshness**: documents re-indexed on change; stale or superseded content still being returned
- **Operations**: latency per stage (embed, search, LLM), cost per answer, user thumbs up/down, top refused questions

## 9. Recap
- RAG = **chunk + embed + store**, then **retrieve + prompt + cite**
- Metadata earns its keep: **security trimming** and **freshness** are filters in the query, not afterthoughts
- A **threshold** lets the system say "I could not find this" instead of guessing
- Cited, numbered context makes answers checkable; offline mode shows the pipeline works without a model
- Retrieval can be measured (Recall@k, refusal rate), but calibrating on the same data is optimistic

**What changes tomorrow (Day 3):** chunking strategies compared, hybrid search (BM25 + vectors), reranking,
proper RAG evaluation, and hallucination mitigation.